# H3a: staranność bio odbiorcy a prawy swipe

Parametryzowany notebook (papermill). Logika: `src/rentfriends/h3a/`. Specyfikacja:
`docs/prompts/h3a-notebook-prompt.md`; decyzje: `docs/h3a/decision-log.md`.

**Prywatność:** notebook wyświetla i zapisuje wyłącznie agregaty (komórki < 5 użytkowników
wytłumione). Tekst bio, `user_key` ani wiersze tabel nie są nigdzie pokazywane.

Wnioski dotyczą **związku**, nie przyczynowości; match ≠ najem.

In [ ]:
# Parameters (papermill)
RUN_LEVEL = 1  # 3 = poziom 3 (diagnostyka, SE dla GLMM, figury)
SAMPLE_FRAC = 1.0
SAMPLE_BY = "sender"
SEED = 20261102
MARGIN = 0.10
FIT_TIMEOUT_MIN = 90
REPORT_DIR = "reports/h3a"
ONLY = ""  # poziom 3: "" | diagnostics | se | figures | lme4
RUN_BOOTSTRAP = 0  # poziom 3, B3 (bootstrap po nadawcach; noc)

In [ ]:
import os
from pathlib import Path

if Path.cwd().name == "notebooks":
    os.chdir("..")

import pandas as pd

from rentfriends.h3a import analysis as A
from rentfriends.h3a.summary import write_summary

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)
LEGACY = not (RUN_LEVEL >= 3 and ONLY)  # ONLY pomija kroki poziomów 1-2
assert SAMPLE_BY == "sender", "only sender-level sampling is implemented"
ctx = A.setup(
    A.Params(
        run_level=RUN_LEVEL,
        sample_frac=SAMPLE_FRAC,
        sample_by=SAMPLE_BY,
        seed=SEED,
        margin=MARGIN,
        fit_timeout_min=FIT_TIMEOUT_MIN,
        report_dir=REPORT_DIR,
    )
)

## 1. Dane, reguła as-of (D7), przepływ próby (F0) i kontrola zgodności z kartą M1

In [ ]:
flow = A.prepare(ctx)
display(flow)
display(pd.DataFrame(ctx.res["card_check"]))
print("Hyderabad:", ctx.res["hyderabad"])
print("Źródło wersji as-of:", ctx.res["asof_source_counts"])
print("G18 (bio_length vs tekst):", ctx.res["g18_bio_length_mismatch"])
print("Braki po filtrze Hyderabad:", ctx.res["missing_by_var_after_hyd"])

## 2. Statystyki opisowe (F1, F2, F8)

In [ ]:
if LEGACY:
    A.describe(ctx)
    print(ctx.res["desc"])

## 3. Sanity check GLMM na danych symulowanych

Rzeczywisty design (X, nadawcy, odbiorcy), znane parametry: β_bio = 0,15, σ²_nadawca = 1,5,
σ²_odbiorca = 0,5. Estymator powinien je odtworzyć, a 95% CI pokrywać prawdę w ~95% powtórzeń.

In [ ]:
if LEGACY:
    A.sanity_check(ctx)

## 4. Modele główne M1 (GLMM), M2 (pooled logit, klastrowanie 2-kier.), M3 (FE nadawcy)
Etykieta według reguły z sekcji 7 promptu (ustalonej przed wynikami).

In [ ]:
if LEGACY:
    main_tab = A.main_models(ctx)
    display(main_tab)
    print(ctx.res["verdict"])
    print(ctx.res["m1_diag"])
    print(ctx.res["m3_diag"])

## 5. M5: długość bio (drugorzędne, bez werdyktu; F5)

In [ ]:
if LEGACY:
    display(A.m5_length(ctx))

## 6. M6: przyrost predykcyjny (opisowy; F7)

In [ ]:
if LEGACY:
    m6 = A.m6_predictive(ctx)
    display({k: v for k, v in m6.items() if not k.startswith("_")})

## 7. M4: analiza w obrębie odbiorcy z bramką wykonalności (F6)

In [ ]:
if LEGACY:
    m4 = A.m4_within(ctx)
    display(m4)

## 8. Analizy wrażliwości S1-S8 (M2 i M3; na poziomie 2 także GLMM dla S1, S3, S4, S7; F4)

In [ ]:
if LEGACY:
    sens = A.sensitivity(ctx, level=1)
    display(sens)
    print(ctx.res["s1_counts"])
    print(ctx.res["s8"]["interaction"])

## 9. Poziom 2: refity GLMM dla S1, S3, S4, S7 i cross-check lme4

In [ ]:
if LEGACY:
    if RUN_LEVEL >= 2:
        sens = A.sensitivity(ctx, level=2)
        display(sens[sens["model"] == "M1"])
    print(A.lme4_crosscheck(ctx))

## 10. Wykres główny (F3) i podsumowanie

In [ ]:
if LEGACY:
    forest_df = A.forest(ctx)
    path = write_summary(ctx.res, REPORT_DIR, RUN_LEVEL)
    print(path.read_text())

## 11. Poziom 3: diagnostyka post hoc (A1-A5), SE dla GLMM (B), figury i eksport (C)

Parametry `ONLY` i `RUN_BOOTSTRAP`; kolejność: A5, figury, A1-A4, B1, B2. Wynik: `reports/h3a/summary_level3.md`, `reports/h3a/export/`.

In [ ]:
if RUN_LEVEL >= 3:
    from rentfriends.h3a import level3 as L3

    files = L3.run_level3(ctx, only=ONLY, bootstrap=bool(RUN_BOOTSTRAP))
    print(files)